# EcoSort AI - Expanded Local Training

The public Garbage265 collection is combined with the original selected 20,000 images. Actual counts come from source indexes and image audits, not the advertised count. All original files and historical results remain separate.

## 1. Project and environment

Confirm the working folder and the native Apple Silicon GPU. This notebook uses the project virtual environment.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import tensorflow as tf
PROJECT = Path.cwd()
assert (PROJECT / 'prepare_expanded.py').exists()
print(tf.config.list_physical_devices('GPU'))

## 2. Download and verify the new source archives

The downloader resumes completed ranges and verifies both complete archives against the source repository SHA-256 values. The original publisher is tany0699 on ModelScope; files are retrieved from the fritz1024 public mirror. This collection contains Chinese item names, not City of Sydney disposal decisions.

In [ ]:
%run download_expanded.py

## 3. Restore required RealWaste originals

Restore images referenced by the old manifest from the authors' repository, verifying every image against its saved SHA-256. Other old source archives already exist under data/.

In [ ]:
%run restore_realwaste.py

## 4. Audit, map categories, deduplicate and split

Decode every candidate, reject small or corrupt images, retain source fine labels, group exact and perceptual duplicates and related source names, exclude conflicting label groups, and create approximately 80/10/10 group-separated partitions. No generated augmentation is counted as a new source image. Near-duplicate detection is heuristic.

In [ ]:
%run prepare_expanded.py

## 5. Inspect actual data coverage

Review class counts and the saved contact sheet before training. Ambiguous material labels are represented as mixed items rather than being silently relabelled as plastic, glass or ceramics.

In [ ]:
report = json.loads((PROJECT / 'data/expanded/dataset_verification.json').read_text())
print(json.dumps(report, indent=2))
display(pd.read_csv(PROJECT / 'data/expanded/class_counts.csv'))
from IPython.display import Image, display
display(Image(filename=str(PROJECT / 'data/expanded/class_review.jpg')))

## 6. Train the full expanded model

The training script first fits a frozen MobileNetV3Large head, then fine-tunes the last 60 backbone layers except BatchNorm with augmentation and a low learning rate. Smoothed, capped training-only class weights reduce extreme compensation. Every training image is visited each epoch, with no subset or steps-per-epoch cap. Validation macro-F1 selects checkpoints. The test partition never updates weights. Full source code is in train_full.py.

In [ ]:
%run train_full.py

## 7. Inspect completed evaluation

The selected model is chosen using validation only. Compare both models on the held-out test partition. Training is complete only when the completion record exists; historical scores are not substituted for this run.

In [ ]:
completion = json.loads((PROJECT / 'trained_expanded/training_completed.json').read_text())
assert completion['status'] == 'completed'
print(json.dumps(completion, indent=2))
display(pd.read_csv(PROJECT / 'trained_expanded/baseline_vs_finetuned.csv'))